# 19.3 scikit-learn ile uçtan uca örnek

*Makine Öğrenmesi Çalışma Rehberi · Bölüm 19: Uygulamada Makine Öğrenmesi*

**Nasıl kullanılır?** Hücreleri sırayla çalıştır (Shift + Enter) ya da menüden *Çalışma zamanı → Tümünü çalıştır*'ı seç. GPU gerekmez. Kodu değiştirip yeniden çalıştırmaktan çekinme. Köşeli parantezli numaralar ([76] gibi) PDF'teki kaynakçaya karşılık gelir.

Aşağıdaki kod, Wisconsin Üniversitesi'nde derlenen meme tümörü teşhis veri setini [73] kullanarak bu rehberdeki ilkeleri bir araya getirir: test kümesini en başta ayırmak, ölçeklemeyle modeli tek boru hattında birleştirmek, çapraz doğrulamayla karşılaştırmak ve test kümesinde tek seferlik değerlendirme yapmak. Veri seti 569 örnek ve 30 özellikten oluşur ve scikit-learn kütüphanesiyle [74] birlikte gelir.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 1) Veriyi yükle: 569 tümör örneği, 30 özellik, 2 sınıf
X, y = load_breast_cancer(return_X_y=True)

# 2) Test kümesini en başta ayır (sınıf oranlarını koruyarak)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

# 3) Ölçekleme + model tek bir boru hattında (veri sızıntısını önler)
modeller = {
    "Lojistik regresyon": make_pipeline(StandardScaler(),
                                        LogisticRegression(max_iter=1000)),
    "Rastgele orman": RandomForestClassifier(n_estimators=200,
                                             random_state=42),
}

# 4) Yalnızca eğitim verisiyle 5-katlı çapraz doğrulama
for ad, model in modeller.items():
    skorlar = cross_val_score(model, X_train, y_train, cv=5, scoring="f1")
    print(f"{ad}: F1 = {skorlar.mean():.3f} +/- {skorlar.std():.3f}")

# 5) Seçilen modeli eğit ve test kümesinde BİR KEZ değerlendir
secilen = modeller["Lojistik regresyon"].fit(X_train, y_train)
print(classification_report(y_test, secilen.predict(X_test), digits=3))

**Beklenen çıktı** (kısaltılmış)

```text
Lojistik regresyon: F1 = 0.984 +/- 0.010
Rastgele orman: F1 = 0.968 +/- 0.015
              precision    recall  f1-score   support

           0      0.976     0.976     0.976        42
           1      0.986     0.986     0.986        72

    accuracy                          0.982       114
```

Çapraz doğrulamada lojistik regresyon F1 ≈ 0,984, rastgele orman F1 ≈ 0,968 verdi; seçilen modelin test kümesindeki doğruluğu 0,982 oldu. (Bu veri setinde sınıf 0 kötü huylu, sınıf 1 iyi huylu tümördür; sayılar kütüphane sürümüne göre çok küçük farklılık gösterebilir.) Basit bir doğrusal modelin burada güçlü bir topluluk yöntemine yetişmesi, hatta onu geçmesi önemli bir dersi hatırlatır: Her zaman önce basit modeli dene.